
# Election Anomaly & Cyber Risk Analysis
### **Author:** Iva Cvetkovic
#### A project that combines cybersecurity and statistics to detect anomalies and potential tampering in precinct-level election data.  
#### This notebook includes: data loading (real NJ precinct file or realistic randomized dataset), cleaning/standardization for the Harvard Dataverse format, anomaly detection at precinct & county level (LOF, heurisics), Benford & last-digit forensic tests, tamper-simulation and tampering index, visualizations), and reproducibility files.
---

Real NJ Dataset citation (only a sample of the dataverse was used):

MIT Election Data and Science Lab, 2022, "U.S. President Precinct-Level Returns 2020", https://doi.org/10.7910/DVN/JXPREB, Harvard Dataverse, V4

    
Registered Votes on County-Level (for the Real NJ Dataset):/https://www.nj.gov/state/elections/assets/pdf/election-results/2020/2020-official-general-voter-turnout.pdf?utm

### To run this program:
1. Upload **NJ_precinct_data.csv** and **real_county_reg.csv** into the Files-content panel.
2. Run the **first** cell to install needed tools.
3. In **Dataset Selection**, select one of the two datasets.
4. Run each cell **in order**.



### **Important Note**:
New Jersey does not publish precinct-level registered voter counts in public data.
Because this value does not exist in the dataset, some forensic tests cannot be performed/applied on real data.

(This limitation is due to data availablity, not methodology.)



The first cell installs and loads the tools needed for the analysis.
It ensures the environment is ready to run the program.

It only needs to be run once.

In [ ]:
#@title Installing needed tools

!pip install colorama folium ipywidgets > /dev/null

import os
import math
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets
import folium
from colorama import Fore, Style

SIG_LEVEL = 0.05
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df_county = None
DATASET_TYPE = None

NJ_DATA_PATH = '/content/NJ_precinct_data.csv'
NJ_DATA_REG = '/content/real_county_reg.csv'

print('Libraries imported.')

The next cell is for choosing which dataset you want to analyze:
 - **Synthetic Dataset** - generated by the program.
 - **Real NJ Dataset** - includes a sample from a real NJ Dataset,  precinct-level vote results with county-level registered voters

 After selecting from the dropdown, a **preview** will appear showing first few rows of the data type you selected.

In [ ]:
#@title Dataset Selection
# Select which dataset you want to analyze, then click "Load selected dataset".

def standardize_and_pivot_noheader(csv_path):
    cols = ['precinct_name', 'office', 'party_detailed', 'party_type', 'votes_type', 'votes',
            'county_name', 'county_code', 'county_name_dup', 'county_code_dup', 'candidate',
            'election_scope', 'office_title', 'year', 'election_type', 'state', 'state_code',
            'flag1', 'flag2', 'state_abbrev', 'district', 'other1', 'other2', 'election_date',
            'flag3', 'sequence']

    df = pd.read_csv(csv_path, header=None, names=cols, dtype=str)

    df = df[['precinct_name', 'county_name', 'party_detailed', 'votes', 'office', 'year']].copy()

    df['votes'] = pd.to_numeric(df['votes'].str.replace(',', ''), errors='coerce').fillna(0).astype(int)

    df.columns = df.columns.str.strip().str.lower()

    party_map = {
        'DEM': 'votes_partyA',
        'DEMOCRAT': 'votes_partyA',
        'REPUBLICAN': 'votes_partyB',
        'GOP': 'votes_partyB'
    }
    df['party_mapped'] = df['party_detailed'].str.upper().map(party_map)

    pivot = df.pivot_table(
        index=['precinct_name', 'county_name'],
        columns='party_mapped',
        values='votes',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    for col in ['votes_partyA', 'votes_partyB']:
        if col not in pivot.columns:
            pivot[col] = 0

    pivot['total_votes'] = pivot['votes_partyA'] + pivot['votes_partyB']

    return pivot[['precinct_name', 'county_name', 'votes_partyA', 'votes_partyB', 'total_votes']]

dataset_dropdown = widgets.Dropdown(
    options=['Use Randomized Synthetic Dataset', 'Use Real NJ Dataverse'],
    value='Use Real NJ Dataverse',
    description='Dataset choice:',
    layout=widgets.Layout(width='70%')
)

load_button = widgets.Button(description='Load selected dataset', button_style='primary')
output = widgets.Output()

def _on_load_clicked(b):
    with output:
        output.clear_output(wait=True)
        choice = dataset_dropdown.value
        global df, df_county, DATASET_TYPE
        try:
            if choice.startswith("Use Randomized"):
                np.random.seed(42)
                precincts = 2000
                counties = [f"County {i}" for i in range(1, 22)]

                benford_probs = np.array([np.log10(1 + 1/d) for d in range(1, 10)])  # Benford distribution for digits 1–9
                first_digits = np.random.choice(np.arange(1, 10), size=precincts, p=benford_probs)

                scales = 10 ** np.random.uniform(2, 3.7, size=precincts)
                total_votes = (first_digits * scales).astype(int)

                df_synth = pd.DataFrame({
                    "precinct_name": [f"Precinct_{i}" for i in range(precincts)],
                    "county": np.random.choice(counties, precincts),
                })

                df_synth["total_votes"] = total_votes

                shareA = np.random.uniform(0.4, 0.6, size=precincts)
                df_synth["votes_partyA"] = (df_synth["total_votes"] * shareA).astype(int)
                df_synth["votes_partyB"] = df_synth["total_votes"] - df_synth["votes_partyA"]

                df_synth["registered_voters"] = (df_synth["total_votes"] * np.random.uniform(1.2, 2.0, size=len(df_synth))).astype(int)

                anomaly_idx = np.random.choice(df_synth.index, size=10, replace=False)
                df_synth.loc[anomaly_idx[:5], "registered_voters"] = (df_synth.loc[anomaly_idx[:5], "total_votes"] * 0.9).astype(int)
                df_synth.loc[anomaly_idx[5:], "registered_voters"] = (df_synth.loc[anomaly_idx[5:], "total_votes"] * 5).astype(int)

                df_synth["turnout"] = df_synth["total_votes"] / df_synth["registered_voters"] * 100

                df = df_synth
                DATASET_TYPE = "random"
                df_county = df.groupby("county", as_index=False).agg({
                    "votes_partyA": "sum",
                    "votes_partyB": "sum",
                    "total_votes": "sum",
                    "registered_voters": "sum"
                })
                df_county["turnout"] = df_county["total_votes"] / df_county["registered_voters"] * 100

                display(Markdown(f"**Loaded randomized synthetic dataset with {len(df)} rows (including anomalies).**"))
                display(df.head())
            else:
                if not os.path.exists(NJ_DATA_PATH):
                    raise FileNotFoundError(f"Dataset not found at: {NJ_DATA_PATH}")

                df = standardize_and_pivot_noheader(NJ_DATA_PATH)
                df['county_name'] = df['county_name'].str.strip().str.upper()

                registration_df = pd.read_csv(NJ_DATA_REG)
                registration_df = registration_df[["county", "registered_voters"]]
                registration_df["county"] = registration_df["county"].str.strip().str.upper()

                df_county = (
                    df.groupby("county_name", as_index=False)
                      .agg({
                          "votes_partyA": "sum",
                          "votes_partyB": "sum",
                          "total_votes": "sum"
                      })
                      .rename(columns={"county_name": "county"})
                )

                df_county = df_county.merge(registration_df, on="county", how="left")

                df_county["turnout"] = df_county["total_votes"] / df_county["registered_voters"] * 100

                DATASET_TYPE = "real"

                display(Markdown(f"**Loaded real NJ dataset with {len(df)} PRECINCT rows (registered voters not in data).**"))
                display(df.head())

                display(Markdown("**County-level view:**"))
                display(df_county.head())

        except FileNotFoundError as e:
            display(Markdown(f"<b style='color:red;'>Error loading dataset:</b> {e}"))


load_button.on_click(_on_load_clicked)

display(dataset_dropdown, load_button, output)

The next step prepares the selected dataset so that later analysis works properly:

In [ ]:
#@title Data Standardization & Cleaning

try:
    df.head()
except NameError:
    raise RuntimeError('No dataset loaded. Please run the dataset selection cell and click "Load selected dataset".')

df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={
    'votes_partya': 'votes_partyA',
    'votes_partyb': 'votes_partyB',
    'county_name': 'county',
})

numeric_cols = ['votes_partyA', 'votes_partyB', 'total_votes']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

if 'total_votes' not in df.columns:
    df['total_votes'] = df['votes_partyA'] + df['votes_partyB']

df = df.dropna(subset=['precinct_name', 'county'])

if 'DATASET_TYPE' in globals() and DATASET_TYPE == "real":
    df['registered_voters'] = np.nan
    df['turnout'] = np.nan
else:
    if 'registered_voters' in df.columns:
        df['turnout'] = (df['total_votes'] / df['registered_voters'].replace(0, np.nan)) * 100
    else:
        df['registered_voters'] = df['total_votes']
        df['turnout'] = 100.0

if 'votes_partyA' in df.columns and 'votes_partyB' in df.columns:
    df['vote_share_diff'] = (df['votes_partyA'] - df['votes_partyB']).abs() / df['total_votes'].replace(0, np.nan) * 100
else:
    df['vote_share_diff'] = np.nan

df['turnout_flag'] = df['turnout'] > 100
df.loc[df['turnout'] > 100, 'turnout'] = 100.0

display(Markdown('### Dataset after standardization — first rows'))
display(df.head(5))
display(Markdown(f"Total precinct rows: {len(df)}"))

Now, the method called **Local Outlier Factor (LOF)** will be used to detect unusual precincts/counties (depending on the dataset chosen).

A place might be flagged as unusual if:
- its turnout is much higher/lower than typical
- its vote share is very lopsided
- it behaves differently from its surrounding

For synthetic data, LOF will detect precinct anomalies.
For the real dataset, it will detect county-level anomalies, since precinct-level turnout is not available.

In [ ]:
#@title LOF Results

try:
    df.head()
except NameError:
    raise RuntimeError('No dataset loaded. Please run the dataset selection cell and click "Load selected dataset".')

if 'DATASET_TYPE' in globals() and DATASET_TYPE == "real":
    if 'df_county' not in globals() or df_county is None:
        raise RuntimeError("df_county is missing for real dataset. Make sure loading cell ran successfully.")

    base = df_county.copy()
    if 'vote_share_diff' not in base.columns:
        base['vote_share_diff'] = (base['votes_partyA'] - base['votes_partyB']).abs() / base['total_votes'].replace(0, np.nan) * 100

    base['turnout_dev'] = base['turnout'] - base['turnout'].mean()

    base['anomaly_score_basic'] = base['turnout_dev'].abs().fillna(0) + base['vote_share_diff'].fillna(0)

    features = base[['turnout_dev', 'vote_share_diff']].fillna(0)

    n_neighbors = min(8, max(2, len(base) - 1))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=0.2)  # allow up to ~20% anomaly counties

    try:
        base['lof_flag'] = lof.fit_predict(features)
        base['lof_flag_bool'] = base['lof_flag'] == -1
    except Exception as e:
        base['lof_flag'] = 1
        base['lof_flag_bool'] = False
        print('County-level LOF failed:', e)

    df_county = base

    display(Markdown('### County-level LOF results (real NJ data)(sample)'))
    display(df_county[['county', 'turnout', 'vote_share_diff', 'anomaly_score_basic', 'lof_flag_bool']].head(10))

    plt.figure(figsize=(10, 4))
    colors = ['red' if x else 'blue' for x in df_county['lof_flag_bool']]
    plt.bar(df_county['county'], df_county['turnout'], color=colors)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Turnout (%)')
    plt.title('County turnout (real NJ data)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.bar(df_county['county'], df_county['turnout_dev'], color=colors)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Turnout deviation from state mean')
    plt.title('County turnout deviation (real NJ data)')
    plt.tight_layout()
    plt.show()

else:
    county_avg_turnout = df.groupby('county')['turnout'].mean()
    df['county_avg_turnout'] = df['county'].map(county_avg_turnout)
    df['turnout_dev'] = df['turnout'] - df['county_avg_turnout']

    df['anomaly_score_basic'] = df['turnout_dev'].abs().fillna(0) + df['vote_share_diff'].fillna(0)

    features = df[['turnout_dev', 'vote_share_diff']].fillna(0)
    n_neighbors = min(20, max(5, int(len(df) * 0.02)))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=0.05)

    try:
        df['lof_flag'] = lof.fit_predict(features)
        df['lof_flag_bool'] = df['lof_flag'] == -1
        df['precinct_num'] = df['precinct_name'].str.extract(r'(\d+)').astype(int)
    except Exception as e:
        df['lof_flag'] = 1
        df['lof_flag_bool'] = False
        print('LOF failed:', e)

    display(Markdown('### Precinct LOF results(sample)(synthetic data)'))
    display(df[['precinct_name', 'county', 'turnout', 'vote_share_diff', 'anomaly_score_basic', 'lof_flag_bool']].head())
    sample = (
    df.groupby(['precinct_name', 'precinct_num'])
      .agg({'turnout': 'mean', 'turnout_dev': 'mean', 'lof_flag_bool': 'first'})
      .reset_index()
      .sort_values('precinct_num')
      .head(40))

    plt.figure(figsize=(12, 4))
    plt.bar(sample['precinct_name'], sample['turnout'])
    plt.xticks(rotation=45, ha='right')
    plt.title('Turnout (sample of precincts)(synthetic data)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    colors = ['red' if x else 'blue' for x in df['lof_flag_bool'].head(40)]
    plt.bar(df['precinct_name'].head(40), df['turnout_dev'].head(40), color=colors)
    plt.xticks(rotation=45, ha='right')
    plt.title('Turnout deviation (first precincts)(synthetic data)')
    plt.tight_layout()
    plt.show()

The next cell gives an overview of all county-level patterns known so far:

In [ ]:
#@title County Summary

try:
    df.head()
except NameError:
    raise RuntimeError('No dataset loaded. Please run the dataset selection cell and click "Load selected dataset".')

global county_summary

if 'DATASET_TYPE' in globals() and DATASET_TYPE == "real" and 'df_county' in globals() and df_county is not None:
    base = df_county.copy()

    if 'lof_flag_bool' in df.columns:
        anom = df.groupby('county')['lof_flag_bool'].mean().reset_index().rename(columns={'lof_flag_bool': 'precinct_anomaly_rate'})
        base = base.merge(anom, on='county', how='left')
    else:
        base['precinct_anomaly_rate'] = np.nan

    base['turnout_dev_state'] = base['turnout'] - base['turnout'].mean()
    county_summary = base

    display(Markdown('### County summary from real NJ data (county-level registration) — sample'))

else:
    county_summary = df.groupby('county').agg({
        'turnout': 'mean',
        'vote_share_diff': 'mean',
        'total_votes': 'sum',
        'precinct_name': 'count',
        'lof_flag_bool': 'mean'
    }).rename(columns={'precinct_name': 'num_precincts', 'lof_flag_bool': 'precinct_anomaly_rate'}).reset_index()

    county_summary['turnout_dev_state'] = county_summary['turnout'] - county_summary['turnout'].mean()

    display(Markdown('### County summary (synthetic or precinct-based) — sample'))

county_feats = county_summary[['turnout_dev_state', 'precinct_anomaly_rate']].fillna(0)
lof_county = LocalOutlierFactor(n_neighbors=min(5, max(2, int(len(county_summary) / 2))), contamination=0.10)
try:
    county_summary['lof_county_flag'] = lof_county.fit_predict(county_feats) == -1
except Exception as e:
    county_summary['lof_county_flag'] = False
    print('County LOF failed:', e)

display(county_summary.head(5))

The next cell tests out the data in multiple ways:
- through last-digit tests
- Benford's Law
- checking for duplicates

It works the best on the synthetic dataset, since it has precent-level data that is needed.

That is because at the county level, data is too small in number and does not span multiple orders of magnitude, so Benford's Law wouldn't be applicable.

In [ ]:
#@title Safety Tests (most applicable on Synthetic)

# tests (Benford first-digit, last-digit rounding, duplicates)

def benford_first_digit_test(series):
    if DATASET_TYPE == "real":
        county_summary["benford_flag"] = False
        county_summary["benford_p"] = np.nan
    else:
        s = series.dropna().astype(int)
        s = s[s != 0]
        if len(s) < 30:
            return {'chi2': np.nan, 'p_value': np.nan, 'obs_freq': None, 'expected': None}
        first_digits = s.astype(str).str.lstrip('-').str.lstrip('0').str[0].astype(int)
        counts = first_digits.value_counts().reindex(range(1, 10), fill_value=0).values
        expected = np.array([np.log10(1 + 1 / d) for d in range(1, 10)])
        chi2_stat, p_value = stats.chisquare(f_obs=counts, f_exp=expected * counts.sum())
        return {'chi2': float(chi2_stat), 'p_value': float(p_value), 'obs_freq': counts / counts.sum(), 'expected': expected}

def last_digit_rounding_checks(series):
    s = series.dropna().astype(int)
    if len(s) == 0:
        return {'chi2_last': np.nan, 'p_last': np.nan, 'prop_mult_5': np.nan,
                'prop_mult_10': np.nan, 'last_freq': None}
    last_digits = (s.astype(str).str[-1].astype(int))
    counts = last_digits.value_counts().reindex(range(10), fill_value=0).values
    expected = np.ones(10) / 10.0
    chi2_stat, p_value = stats.chisquare(f_obs=counts, f_exp=expected * counts.sum())
    prop_mult_5 = (last_digits % 5 == 0).mean()
    prop_mult_10 = (last_digits % 10 == 0).mean()
    return {'chi2_last': float(chi2_stat), 'p_last': float(p_value),
            'prop_mult_5': float(prop_mult_5), 'prop_mult_10': float(prop_mult_10),
            'last_freq': counts / counts.sum()}

def duplicate_counts_check(series):
    s = series.dropna().astype(int)
    vc = s.value_counts()
    num_duplicate_values = int((vc > 1).sum())
    num_total_duplicates = int(vc[vc > 1].sum())
    top_repeats = vc.head(10)
    return {
        'num_duplicate_values': num_duplicate_values,
        'num_total_duplicates': num_total_duplicates,
        'top_repeats': top_repeats
    }

# precinct or county level the question is
if 'DATASET_TYPE' in globals() and DATASET_TYPE == "real" and 'df_county' in globals() and df_county is not None:
    target = df_county
    level = 'county'
    display(Markdown('### Forensic tests: county level (real NJ datas)'))
else:
    target = df
    level = 'precinct'
    display(Markdown('### Forensic tests: precinct level (synthetic dataset)'))

benford_result = benford_first_digit_test(target['total_votes'])
lastdigit_result = last_digit_rounding_checks(target['total_votes'])
duplicates_result = duplicate_counts_check(target['total_votes'])

if 'turnout' in target.columns:
    target['turnout_z_state'] = stats.zscore(target['turnout'], nan_policy='omit')
    target['turnout_z_state'] = target['turnout_z_state'].fillna(0)
    high_z = target[np.abs(target['turnout_z_state']) > 3]
    num_high_z = high_z.shape[0]
else:
    num_high_z = 0

if DATASET_TYPE == "real":
    display(Markdown(f"Benford χ²=not aplicable"))
    display(Markdown(f"Last-digit χ²={lastdigit_result['chi2_last']:.2f}, p-value={lastdigit_result['p_last']:.4f}"))
    display(Markdown(f"Duplicate distinct repeated vote-count values: {duplicates_result['num_duplicate_values']}"))
    display(Markdown(f"{level.capitalize()} units with |turnout z|>3: {num_high_z}"))

else:
    display(Markdown(f"Benford χ²={benford_result['chi2']:.2f}, p-value={benford_result['p_value']:.4f}"))
    display(Markdown(f"Last-digit χ²={lastdigit_result['chi2_last']:.2f}, p-value={lastdigit_result['p_last']:.4f}"))
    display(Markdown(f"Duplicate distinct repeated vote-count values: {duplicates_result['num_duplicate_values']}"))
    display(Markdown(f"{level.capitalize()} units with |turnout z|>3: {num_high_z}"))

The next cell gives us a summary of what was computed in the previous cell for the synthetic dataset (if chosen):

In [ ]:
#@title Forensic Flags (Synthetic only)

# county forensic tests (only where enough precincts or using county lvl df)

# We still want benford_flag / last_flag columns so that the tampering index works,
# but we avoid running per-precinct Benford tests for each county when using real data.

if 'DATASET_TYPE' in globals() and DATASET_TYPE == "real" and 'df_county' in globals() and df_county is not None:
    # Ensure the necessary columns exist; default to no flags (0) so the score is driven by turnout + anomalies
    if 'benford_flag' not in county_summary.columns:
        county_summary['benford_flag'] = False
        county_summary['benford_p'] = np.nan
    if 'last_flag' not in county_summary.columns:
        county_summary['last_flag'] = False
        county_summary['last_p'] = np.nan
else:
    county_benford = []
    county_lastdigit = []
    for cname, group in df.groupby('county'):
        if len(group) < 30:
            county_benford.append({'county': cname, 'benford_p': np.nan, 'benford_flag': False})
            county_lastdigit.append({'county': cname, 'last_p': np.nan, 'last_flag': False})
            continue

        bf = benford_first_digit_test(group['total_votes'])
        ld = last_digit_rounding_checks(group['total_votes'])

        county_benford.append({
            'county': cname,
            'benford_p': bf['p_value'],
            'benford_flag': (bf['p_value'] < SIG_LEVEL if not np.isnan(bf['p_value']) else False)
        })

        county_lastdigit.append({
            'county': cname,
            'last_p': ld['p_last'],
            'last_flag': (ld['p_last'] < SIG_LEVEL if not np.isnan(ld['p_last']) else False)
        })

    county_bf_df = pd.DataFrame(county_benford).set_index('county')
    county_ld_df = pd.DataFrame(county_lastdigit).set_index('county')

    for col in ['benford_p', 'benford_flag', 'last_p', 'last_flag']:
        if col in county_summary.columns:
             county_summary = county_summary.drop(columns=[col])

    county_summary = (
         county_summary
             .set_index('county')
             .join(county_bf_df)
             .join(county_ld_df)
             .reset_index()
        )

    county_summary['benford_flag'] = county_summary['benford_flag'].astype('boolean').fillna(False)
    county_summary['last_flag'] = county_summary['last_flag'].astype('boolean').fillna(False)

if DATASET_TYPE == 'real':
  display(Markdown('### County-level forensic flags:'))
  display(Markdown('Not aplicable for Real NJ Data.'))

else:
  display(Markdown('### County-level forensic flags'))
  cols_to_show = [c for c in ['county', 'num_precincts', 'precinct_anomaly_rate', 'benford_flag', 'last_flag'] if c in county_summary.columns]
  display(county_summary[cols_to_show].head(21))

The Risk Scoring section will now combine several results from previous calculations into a risk score.

Counties are assigned labels based on the level of their risk, however these labels do **not** indicate fraud.

These scores could be used to check which counties might need a deeper investigation.

In [ ]:
#@title Risk Scoring

# Tampering index (alert levels)

scaler = MinMaxScaler()
norm_cols = ['turnout_dev_state','precinct_anomaly_rate']
county_summary[['turnout_norm','precinct_anom_norm']] = scaler.fit_transform(county_summary[norm_cols].fillna(0))

county_summary['benford_int'] = county_summary['benford_flag'].astype(int)
county_summary['last_int'] = county_summary['last_flag'].astype(int)

county_summary['tampering_score'] = (
    0.35 * county_summary['precinct_anom_norm'] +
    0.30 * county_summary['turnout_norm'] +
    0.20 * county_summary['benford_int'] +
    0.15 * county_summary['last_int']
)

def alert_level(score):
    if pd.isna(score):
        return 'UNKNOWN'
    if score > 0.75:
        return 'CRITICAL'
    elif score > 0.6:
        return 'HIGH'
    elif score > 0.4:
        return 'MEDIUM'
    elif score > 0.2:
        return 'LOW'
    return 'NORMAL'

county_summary['tamper_alert'] = county_summary['tampering_score'].apply(alert_level)
display(Markdown('### Tampering score (top counties)'))
display(county_summary.sort_values('tampering_score', ascending=False).head(5))

In [ ]:
#@title Risk Scoring - Graph

# Visual ranking with colors

color_map = {
    'CRITICAL': '#FF4C4C',
    'HIGH': '#FF944C',
    'MEDIUM': '#FFD24C',
    'LOW': '#4CFF88',
    'NORMAL': '#B0C4DE',
    'UNKNOWN': '#AAAAAA'
}

ranked = county_summary.sort_values('tampering_score', ascending=True)

plt.figure(figsize=(10, 8))
colors = [color_map.get(x, '#AAAAAA') for x in ranked['tamper_alert']]
plt.barh(ranked['county'], ranked['tampering_score'], color=colors)
plt.xlabel('Tampering Score (0-1)')

if 'DATASET_TYPE' in globals():
    if DATASET_TYPE == "real":
        plt.title('County Tampering Score (lower = less risk) — REAL NJ DATA')
    else:
        plt.title('County Tampering Score (lower = less risk) — RANDOMIZED SYNTHETIC DATA')
else:
    plt.title('County Tampering Score')

plt.gca().invert_yaxis()

import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=color_map[k], label=k) for k in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'NORMAL']]
plt.legend(handles=patches, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

To demonstrate how anomaly detection responds to manipulated data, this step alters some precincts in the synthetic dataset (if it was chosen) and then measures:
- **Precision**: How many flagged anomalies were truly manipulated
- **Recall**: How many tampered precincts were successfully detected

Results may vary each time the cell is ran because:
- different numbers of manipulated
precincts may be selected
- the size of vote changes varies
- LOF is sensitive to new patterns in the data

This is expected and reflects how anomaly detection behaves in real audits.

In [ ]:
#@title Tamper Simulation (Synthetic only)

from sklearn.metrics import precision_score, recall_score

if 'DATASET_TYPE' in globals() and DATASET_TYPE == "real":
    display(Markdown("_Tamper simulation is only run for the synthetic dataset (precinct-level turnout is unknown for the real data)._"))
else:
    sim = df.copy()

    max_tamper = min(10, len(sim))
    min_tamper = min(5, max_tamper)
    if max_tamper <= 0:
        display(Markdown("_Not enough precincts to run tamper simulation._"))
    else:
        num_tamper = np.random.randint(min_tamper, max_tamper + 1)
        tamper_rate = num_tamper / len(sim)

        tamper_idxs = np.random.choice(sim.index, size=num_tamper, replace=False)
        sim['tampered'] = False
        sim.loc[tamper_idxs, 'tampered'] = True

        for idx in tamper_idxs:
            if 'votes_partyA' in sim.columns:
                sim.at[idx, 'votes_partyA'] = int(
                    sim.at[idx, 'votes_partyA'] * np.random.uniform(1.3, 1.8)
                    + np.random.randint(5, 20))

            if 'votes_partyB' in sim.columns:
                sim.at[idx, 'votes_partyB'] = max(
                    0,
                    int(sim.at[idx, 'votes_partyB'] * np.random.uniform(0.7, 0.95)))

            if 'votes_partyA' in sim.columns and 'votes_partyB' in sim.columns:
                sim.at[idx, 'total_votes'] = sim.at[idx, 'votes_partyA'] + sim.at[idx, 'votes_partyB']

        if 'registered_voters' in sim.columns:
            sim['turnout'] = sim['total_votes'] / sim['registered_voters'].replace(0, np.nan) * 100

        sim['vote_share_diff'] = (
            sim.get('votes_partyA', 0) - sim.get('votes_partyB', 0)
        ).abs() / sim['total_votes'].replace(0, np.nan)

        sim['turnout_dev'] = sim['turnout'] - sim['turnout'].mean()
        sim_features = sim[['turnout_dev', 'vote_share_diff']].fillna(0)

        lof_sim = LocalOutlierFactor(
            n_neighbors=min(20, max(5, int(len(sim) * 0.02))),
            contamination=0.015)

        try:
            sim['lof_flag_sim'] = lof_sim.fit_predict(sim_features) == -1
        except Exception as e:
            sim['lof_flag_sim'] = False
            print("LOF simulation failed:", e)

        precision = precision_score(
            sim['tampered'].astype(int),
            sim['lof_flag_sim'].astype(int),
            zero_division=0)

        recall = recall_score(
            sim['tampered'].astype(int),
            sim['lof_flag_sim'].astype(int),
            zero_division=0)

        display(Markdown(
            f"**Tamper simulation:** injected {num_tamper} tampered precincts (randomly chosen)."))
        display(Markdown(
            f"LOF detection — precision: {precision:.3f}, recall: {recall:.3f}"))

**The goal isn't perfect accuracy, it's to show**:
- how precision and recall must be interpreted in forensic analysis
- why multiple tools are required for precision
- and that this project represents an introductory model rather than a full election security system, since real world tampering detection requires richer data sources and far more complex patterns than what is available in public vote totals

In [ ]:
#@title Results

import datetime

if 'DATASET_TYPE' not in globals():
    raise RuntimeError("Please load a dataset and run the analysis before this summary cell.")

if 'df' not in globals() or 'county_summary' not in globals():
    raise RuntimeError("df or county_summary is missing. Make sure the main analysis cells have been run.")

run_time = datetime.datetime.now().strftime("%Y-%m-%d")

try:
    num_precincts = len(df)
except Exception:
    num_precincts = None

try:
    if 'county' in county_summary.columns:
        num_counties = county_summary['county'].nunique()
    else:
        num_counties = None
except Exception:
    num_counties = None

if (
    'tampering_score' in county_summary.columns
    and 'tamper_alert' in county_summary.columns
    and 'county' in county_summary.columns
):
    top = county_summary.sort_values('tampering_score', ascending=False).head(3)
    top_lines = []
    for _, row in top.iterrows():
        cname = str(row['county'])
        score = float(row['tampering_score'])
        level = str(row['tamper_alert'])
        top_lines.append(f"- **{cname}** — score `{score:.3f}`, alert **{level}**")
    top_md = "\n".join(top_lines) if len(top_lines) > 0 else "- (no counties scored)"
else:
    top_md = "- (tampering score not available in this run)"

if DATASET_TYPE == "real":
    dataset_text = (
        "**Dataset:** Real New Jersey election data\n"
        f"- Counties analyzed: **{num_counties if num_counties is not None else 'unknown'}**\n"
        "- Uses official precinct vote totals and county-level registered voter counts\n"
        "- Precinct-level turnout is not available, so analysis focuses on county patterns\n"
        "- Benford and last-digit tests are turned off here because precinct-level turnout isn't available as well.\n"
    )
    sim_text = "Tamper simulation is **not** run on the real dataset in this notebook."

else:
    dataset_text = (
        "**Dataset:** Synthetic precinct-level dataset\n"
        f"- Precincts analyzed: **{num_precincts if num_precincts is not None else 'unknown'}**\n"
        f"- Counties analyzed: **{num_counties if num_counties is not None else 'unknown'}**\n"
        "- Includes intentionally injected anomalies to test the detection pipeline\n"
        "- Benford-style digit checks and last-digit tests are shown for demonstration\n"
    )
    if 'precision' in globals() and 'recall' in globals():
        sim_text = (
            "Synthetic tampering simulation was run.\n"
            f"- Precision: `{precision:.3f}`\n"
            f"- Recall: `{recall:.3f}`\n"
            "\nThese numbers change each run because different precincts and different tampering sizes are chosen randomly."
        )
    else:
        sim_text = (
            "A synthetic tampering simulation can be run to show how LOF reacts to manipulated precincts. "
            "Precision and recall vary each run due to randomness."
        )

limitations_text = (
    "- This is only an **introductory anomaly-analysis model**.\n"
    "- Real audits require logs, metadata, timestamps, and chain-of-custody information.\n"
    "- LOF anomaly detection changes as the data distribution changes.\n"
    "- Synthetic tampering is simplified for clarity.\n"
)

summary_md = f"""
# Election Anomaly Summary Report
*Generated on {run_time}*

---

## Dataset Overview
{dataset_text}

Precinct rows analyzed: **{num_precincts if num_precincts is not None else 'unknown'}**
Counties in summary: **{num_counties if num_counties is not None else 'unknown'}**

---

## Highest-Risk Counties in This Run
{top_md}
#### (These are statistical irregularities only — not evidence of any wrongdoing.)
---

## Simulation Notes
{sim_text}

---

## Key Limitations
{limitations_text}

---

## Closing Note
### This notebook is an **entry-level demonstration** of how data science and cybersecurity concepts
can be combined to analyze election data. It highlights unusual patterns and shows how
statistical methods can support real-world investigations.
"""

display(Markdown(summary_md))